In [1]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, KNNImputer, MissingIndicator
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder, MinMaxScaler, PowerTransformer, OrdinalEncoder
from sklearn.model_selection import train_test_split
from pathlib import Path

In [5]:
import dagshub
dagshub.init(repo_owner='AMR-ITH', repo_name='RealEstateInsights', mlflow=True)
import mlflow

Accessing as AMR-ITH

Initialized MLflow to track repo "AMR-ITH/RealEstateInsights"

Repository AMR-ITH/RealEstateInsights initialized!

In [6]:
# set the tracking server

mlflow.set_tracking_uri("https://dagshub.com/AMR-ITH/RealEstateInsights.mlflow")

In [7]:
# mlflow experiment

mlflow.set_experiment("Exp 1 - simple model with and without target transformation")

<Experiment: artifact_location='mlflow-artifacts:/3228cb460f7c40c4996e00010eed0095', creation_time=1746439877628, experiment_id='1', last_update_time=1746439877628, lifecycle_stage='active', name='Exp 1 - simple model with and without target transformation', tags={}>

# Load the data

In [25]:
# pathlib is a module in Python that provides an object-oriented interface 
# for working with file system paths.
current = Path.cwd()
parent = current.parent

# load the dat 
df = pd.read_csv(parent /'data-scraped/interim_data.csv')
df.head()



,apartment_name,appartment_loc,zone,bhk_type,construction_status,carpet_area,bulit_area,super_bulit_area,price_value,nearbylocation,facility,luxury_facility_scores
0,nambiar millennia,Sarjapur Road,east,1,Under Construction,NaN,668.0,NaN,0.53,"[('mahatma vidhyalaya', '400 m'), ('eterssrt m...","['Yoga/Meditation Area', ""Children's Play Area...",69
1,provident capella,Samethanahalli,east,1,New Property,NaN,431.0,480.0,0.55,"[('soukya road', '1.4 km'), ('mvj college of e...","[""Children's Play Area"", 'Creche/Day Care', 'J...",70
2,brigade citre budigere cross,Byrathi,east,1,New Property,NaN,619.0,689.0,0.69,"[('one world international school', '3.2kms'),...","['Pet Park', ""Children's Play Area"", 'Landscap...",50
3,sattva east crest bandapura,Budigere Cross,east,1,New Property,NaN,537.0,598.0,0.70,"[('prerana international school', '700 m'), ('...","['Banquet Hall', 'Creche/Day Care', ""Children'...",74
4,sowparnika columns,Soukya Road,east,1,New Property,486.0,619.0,736.0,0.52,"['Whitefield Kadugodi Metro Station', 'Nexus S...","['Lift(s)', 'Swimming Pool', 'Park', 'Fitness ...",44


In [26]:
# check for missing values

df.isnull().sum()

apartment_name               0
appartment_loc               0
zone                         0
bhk_type                     0
construction_status          0
carpet_area               2620
bulit_area                   0
super_bulit_area          2381
price_value                  0
nearbylocation             281
facility                   478
luxury_facility_scores       0
dtype: int64

In [27]:
def categorize_luxury(score):
    if 0 <= score < 50:
        return 'low'
    elif 50 <= score < 150:
        return 'medium'
    else:
        return 'high'

In [28]:
df['luxury_category'] = df['luxury_facility_scores'].apply(categorize_luxury)

In [29]:
df.drop(columns=['carpet_area','super_bulit_area','nearbylocation','facility','apartment_name','appartment_loc','luxury_facility_scores'], inplace=True)

# with out transforming the traget variable : price_value 

In [30]:
temp_df = df.copy()

X = temp_df.drop(columns=['price_value'])
y = temp_df['price_value']

In [31]:
# train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [32]:
print("The size of train data is",X_train.shape)
print("The shape of test data is",X_test.shape)

The size of train data is (4860, 5)
The shape of test data is (1215, 5)


In [38]:
# do the basic processing input data

num_cols = ['bulit_area']
nomial_cols = ['zone']
ordinal_cols = ['construction_status','bhk_type','luxury_category']


In [39]:
for col in ordinal_cols:
    print(col,":",X_train[col].unique())

construction_status : ['New Property' 'Relatively New' 'Under Construction' 'undefined' 'Old'
 'Moderatly Old']
bhk_type : [3 2 4 1]
luxury_category : ['medium' 'low' 'high']


In [40]:
construction_status_order = ['New Property','Under Construction', 'Relatively New', 'Moderatly Old', 'Old','undefined']
bhk_type_order = ['1','2','3','4','5','6','7','8','9','10']
luxury_category_order = ['low','medium','high']

In [41]:
# build a preprocessor

prepocessor = ColumnTransformer(transformers=[
    ("scale", MinMaxScaler(), num_cols),
        ("nominal_encode", OneHotEncoder(handle_unknown="ignore",sparse_output=False), nomial_cols),
    ("ordinal_encode", OrdinalEncoder(categories=[construction_status_order,bhk_type_order,luxury_category_order]), ordinal_cols)
],remainder="passthrough",n_jobs=-1,force_int_remainder_cols=False,verbose_feature_names_out=False)

prepocessor.set_output(transform="pandas")

ColumnTransformer(force_int_remainder_cols=False, n_jobs=-1,
                  remainder='passthrough',
                  transformers=[('scale', MinMaxScaler(), ['bulit_area']),
                                ('nominal_encode',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['zone']),
                                ('ordinal_encode',
                                 OrdinalEncoder(categories=[['New Property',
                                                             'Under '
                                                             'Construction',
                                                             'Relatively New',
                                                             'Moderatly Old',
                                                             'Old',
                                                             'undefined'],
                                                            ['1', '2', '3', '4',
                                                             '5', '6', '7', '8',
                                                             '9', '10'],
                                                            ['low', 'medium',
                                                             'high']]),
                                 ['construction_status', 'bhk_type',
                                  'luxury_category'])],
                  verbose_feature_names_out=False)

In [42]:
# transform the data

X_train_trans = prepocessor.fit_transform(X_train)
X_test_trans = prepocessor.transform(X_test)

X_train_trans

,bulit_area,zone_east,zone_north,zone_south,zone_west,construction_status,bhk_type,luxury_category
2842,0.197885,0.0,1.0,0.0,0.0,0.0,2.0,1.0
903,0.185153,1.0,0.0,0.0,0.0,2.0,2.0,1.0
3262,0.297475,0.0,1.0,0.0,0.0,2.0,2.0,1.0
109,0.114804,1.0,0.0,0.0,0.0,1.0,1.0,1.0
5602,0.092361,0.0,0.0,0.0,1.0,5.0,1.0,0.0
...,...,...,...,...,...,...,...,...
3772,0.224860,0.0,1.0,0.0,0.0,0.0,3.0,1.0
5191,0.140160,0.0,0.0,1.0,0.0,0.0,2.0,1.0
5226,0.175874,0.0,0.0,1.0,0.0,4.0,2.0,1.0
5390,0.118041,0.0,0.0,0.0,1.0,2.0,1.0,1.0


## Linear Regression model

In [43]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# linear regression model
lin_reg = LinearRegression()

lin_reg.fit(X_train_trans, y_train)

# get the prediction
y_pred_train = lin_reg.predict(X_train_trans)
y_pred_test = lin_reg.predict(X_test_trans)

# get the metrics r2,mse,mae
mae_train = mean_absolute_error(y_train, y_pred_train)
mse_train= mean_squared_error(y_train, y_pred_train)
r2_score_train= r2_score(y_train, y_pred_train)

mae_test = mean_absolute_error(y_test, y_pred_test)
mse_test = mean_squared_error(y_test, y_pred_test)
r2_score_test = r2_score(y_test, y_pred_test)


# metrics for linear regression model
print("Linear Regression Model")
print("R2 Train:",r2_score_train)
print("R2 Test:",r2_score_test)
print("MAE Train:",mae_train)
print("MAE Test:",mae_test)

Linear Regression Model
R2 Train: 0.763961282965934
R2 Test: 0.7766655553591342
MAE Train: 0.5953000818061609
MAE Test: 0.5636450729295334


In [44]:
# calculate the cross val score

from sklearn.model_selection import cross_val_score

scores_cv_linear_model = cross_val_score(lin_reg,X_train_trans,y_train,cv=5,scoring="r2",n_jobs=-1)

scores_cv_linear_model

array([0.78619764, 0.79107713, 0.72992827, 0.77905935, 0.72962579])

In [45]:
# log experiment
with mlflow.start_run(run_name="No Target tranformation-lr"):
    # mlflow log exp type
    mlflow.log_param("experiment_type", "no target transformation")
    # log model params
    mlflow.log_params(lin_reg.get_params())
    # log metrics
    mlflow.log_metric("train_r2", r2_score_train)
    mlflow.log_metric("test_r2", r2_score_test)
    mlflow.log_metric("train_mae", mae_train)   
    mlflow.log_metric("test_mae", mae_test)
    # log cross val score
    mlflow.log_metric("cross_val_score", scores_cv_linear_model.mean())

🏃 View run No Target tranformation-lr at: https://dagshub.com/AMR-ITH/RealEstateInsights.mlflow/#/experiments/1/runs/28f691212f2a49788984a6e44d181ebd
🧪 View experiment at: https://dagshub.com/AMR-ITH/RealEstateInsights.mlflow/#/experiments/1


## Random Forest Regressor Model

In [46]:
from sklearn.ensemble import RandomForestRegressor

# random forest model
rf_reg = RandomForestRegressor(n_estimators=100, random_state=42)

rf_reg.fit(X_train_trans, y_train)

# get the prediction
y_pred_train = rf_reg.predict(X_train_trans)
y_pred_test = rf_reg.predict(X_test_trans)

# get the metrics r2,mse,mae
mae_train = mean_absolute_error(y_train, y_pred_train)
mse_train= mean_squared_error(y_train, y_pred_train)
r2_score_train= r2_score(y_train, y_pred_train)

mae_test = mean_absolute_error(y_test, y_pred_test)
mse_test = mean_squared_error(y_test, y_pred_test)
r2_score_test = r2_score(y_test, y_pred_test)


# metrics for linear regression model
print("Random Forest Model")
print("R2 Train:",r2_score_train)
print("R2 Test:",r2_score_test)
print("MAE Train:",mae_train)
print("MAE Test:",mae_test)



Random Forest Model
R2 Train: 0.965841925096
R2 Test: 0.8234646796883452
MAE Train: 0.20082580219095328
MAE Test: 0.46426907712343884


In [47]:
scores_cv_rf_model = cross_val_score(rf_reg,X_train_trans,y_train,cv=5,scoring="r2",n_jobs=-1)

scores_cv_rf_model

array([0.79975701, 0.78221612, 0.74496631, 0.80373468, 0.78555527])

In [48]:
# feature importance plot

(
    pd.DataFrame(
        rf_reg.feature_importances_,
        index=X_train_trans.columns,
        columns=["Feature Importance"]
    )
    .sort_values(by="Feature Importance",ascending=False)
)

,Feature Importance
bulit_area,0.883339
construction_status,0.033392
luxury_category,0.033149
bhk_type,0.015910
zone_west,0.013302
zone_east,0.008832
zone_north,0.006985
zone_south,0.005092


In [49]:
# log experiment
with mlflow.start_run(run_name="No Target tranformation-rf"):
    # mlflow log exp type
    mlflow.log_param("experiment_type", "no target transformation")
    # log model params
    mlflow.log_params(rf_reg.get_params())
    # log metrics
    mlflow.log_metric("train_r2", r2_score_train)
    mlflow.log_metric("test_r2", r2_score_test)
    mlflow.log_metric("train_mae", mae_train)   
    mlflow.log_metric("test_mae", mae_test)
    # log cross val score
    mlflow.log_metric("cross_val_score", scores_cv_rf_model.mean())

🏃 View run No Target tranformation-rf at: https://dagshub.com/AMR-ITH/RealEstateInsights.mlflow/#/experiments/1/runs/df0491e3e56049f4800de9066e191fb5
🧪 View experiment at: https://dagshub.com/AMR-ITH/RealEstateInsights.mlflow/#/experiments/1


# with  transforming the traget variable : price_value 

In [50]:

from sklearn.preprocessing import PowerTransformer

# Setup power transformer
pt = PowerTransformer(method="yeo-johnson")
df["price_value_pt"] = pt.fit_transform(df[["price_value"]])



In [51]:
temp_df = df.copy()

X = temp_df.drop(columns=['price_value','price_value_pt'])
y = temp_df['price_value']  # Original target
y_pt = temp_df['price_value_pt']  # Transformed target

In [52]:
X_train, X_test, y_train, y_test, y_train_pt, y_test_pt = train_test_split(
    X, y, y_pt, test_size=0.2, random_state=42
)

In [53]:
# Check for NaN values in transformed features
print("NaN values in X_train_trans:", np.isnan(X_train_trans).sum())
print("NaN values in X_test_trans:", np.isnan(X_test_trans).sum())

NaN values in X_train_trans: bulit_area             0
zone_east              0
zone_north             0
zone_south             0
zone_west              0
construction_status    0
bhk_type               0
luxury_category        0
dtype: int64
NaN values in X_test_trans: bulit_area             0
zone_east              0
zone_north             0
zone_south             0
zone_west              0
construction_status    0
bhk_type               0
luxury_category        0
dtype: int64


In [54]:
# transform the data

X_train_trans = prepocessor.fit_transform(X_train)
X_test_trans = prepocessor.transform(X_test)

X_train_trans

,bulit_area,zone_east,zone_north,zone_south,zone_west,construction_status,bhk_type,luxury_category
2842,0.197885,0.0,1.0,0.0,0.0,0.0,2.0,1.0
903,0.185153,1.0,0.0,0.0,0.0,2.0,2.0,1.0
3262,0.297475,0.0,1.0,0.0,0.0,2.0,2.0,1.0
109,0.114804,1.0,0.0,0.0,0.0,1.0,1.0,1.0
5602,0.092361,0.0,0.0,0.0,1.0,5.0,1.0,0.0
...,...,...,...,...,...,...,...,...
3772,0.224860,0.0,1.0,0.0,0.0,0.0,3.0,1.0
5191,0.140160,0.0,0.0,1.0,0.0,0.0,2.0,1.0
5226,0.175874,0.0,0.0,1.0,0.0,4.0,2.0,1.0
5390,0.118041,0.0,0.0,0.0,1.0,2.0,1.0,1.0


In [55]:
pt.lambdas_

array([-0.88433093])

In [56]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Linear regression model
lin_reg = LinearRegression()

# Fit on transformed features and transformed target
lin_reg.fit(X_train_trans, y_train_pt)

# Get the predictions (in transformed scale)
y_pred_train_pt = lin_reg.predict(X_train_trans)
y_pred_test_pt = lin_reg.predict(X_test_trans)

# Convert predictions back to original scale
# Need to reshape to 2D array for inverse_transform
y_pred_train = pt.inverse_transform(y_pred_train_pt.reshape(-1, 1)).flatten()
y_pred_test = pt.inverse_transform(y_pred_test_pt.reshape(-1, 1)).flatten()

# Calculate metrics using original scale values
# Compare predictions with actual values in the same scale
mae_train = mean_absolute_error(y_train, y_pred_train)
mse_train = mean_squared_error(y_train, y_pred_train)
r2_score_train = r2_score(y_train, y_pred_train)

mae_test = mean_absolute_error(y_test, y_pred_test)
mse_test = mean_squared_error(y_test, y_pred_test)
r2_score_test = r2_score(y_test, y_pred_test)

# Print metrics for linear regression model
print("Linear Regression Model")
print("R2 Train:", r2_score_train)
print("R2 Test:", r2_score_test)
print("MAE Train:", mae_train)
print("MAE Test:", mae_test)
print("MSE Train:", mse_train)
print("MSE Test:", mse_test)
print("RMSE Train:", np.sqrt(mse_train))
print("RMSE Test:", np.sqrt(mse_test))

c:\Python\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but PowerTransformer was fitted with feature names
  warnings.warn(
c:\Python\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but PowerTransformer was fitted with feature names
  warnings.warn(


ValueError: Input contains NaN.

yeo-johnson method data is highly skewed with extreme outliers.
inverse transform and leads to NaN, inf, or totally unrealistic values
doing other quantile transform methods 

## quantile method

In [57]:
from sklearn.preprocessing import QuantileTransformer

# Setup QuantileTransformer (output distribution set to 'normal')
qt = QuantileTransformer(output_distribution='normal', random_state=42)
df["price_value_pt"] = qt.fit_transform(df[["price_value"]])

temp_df = df.copy()

X = temp_df.drop(columns=['price_value','price_value_pt'])
y = temp_df['price_value']  # Original target
y_pt = temp_df['price_value_pt']  # Transformed target

X_train, X_test, y_train, y_test, y_train_pt, y_test_pt = train_test_split(
    X, y, y_pt, test_size=0.2, random_state=42
)

X_train_trans = prepocessor.fit_transform(X_train)
X_test_trans = prepocessor.transform(X_test)




In [58]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Linear regression model
lin_reg = LinearRegression()

# Fit on transformed features and transformed target
lin_reg.fit(X_train_trans, y_train_pt)



LinearRegression()

In [59]:
# Get the predictions (in transformed scale)
y_pred_train_pt = lin_reg.predict(X_train_trans)
y_pred_test_pt = lin_reg.predict(X_test_trans)

# Convert predictions back to original scale
y_pred_train = qt.inverse_transform(pd.DataFrame(y_pred_train_pt, columns=["price_value"])).flatten()
y_pred_test = qt.inverse_transform(pd.DataFrame(y_pred_test_pt, columns=["price_value"])).flatten()

# Calculate metrics using original scale values
mae_train = mean_absolute_error(y_train, y_pred_train)
mse_train = mean_squared_error(y_train, y_pred_train)
r2_score_train = r2_score(y_train, y_pred_train)

mae_test = mean_absolute_error(y_test, y_pred_test)
mse_test = mean_squared_error(y_test, y_pred_test)
r2_score_test = r2_score(y_test, y_pred_test)

# Print metrics
print("Linear Regression Model")
print("R2 Train:", r2_score_train)
print("R2 Test:", r2_score_test)
print("MAE Train:", mae_train)
print("MAE Test:", mae_test)
print("MSE Train:", mse_train)
print("MSE Test:", mse_test)
print("RMSE Train:", np.sqrt(mse_train))
print("RMSE Test:", np.sqrt(mse_test))

Linear Regression Model
R2 Train: 0.6688678911020504
R2 Test: 0.6881950525480173
MAE Train: 0.6287374261202711
MAE Test: 0.6045534424394925
MSE Train: 1.1564665065101019
MSE Test: 1.0380249720445196
RMSE Train: 1.0753913271503086
RMSE Test: 1.0188351054240914


In [60]:

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Linear regression model
rf_reg = RandomForestRegressor()

# Fit on transformed features and transformed target
rf_reg.fit(X_train_trans, y_train_pt)

RandomForestRegressor()

In [61]:
# Get the predictions (in transformed scale)
y_pred_train_pt = rf_reg.predict(X_train_trans)
y_pred_test_pt = rf_reg.predict(X_test_trans)

# Convert predictions back to original scale
y_pred_train = qt.inverse_transform(pd.DataFrame(y_pred_train_pt, columns=["price_value"])).flatten()
y_pred_test = qt.inverse_transform(pd.DataFrame(y_pred_test_pt, columns=["price_value"])).flatten()

# Calculate metrics using original scale values
mae_train = mean_absolute_error(y_train, y_pred_train)
mse_train = mean_squared_error(y_train, y_pred_train)
r2_score_train = r2_score(y_train, y_pred_train)

mae_test = mean_absolute_error(y_test, y_pred_test)
mse_test = mean_squared_error(y_test, y_pred_test)
r2_score_test = r2_score(y_test, y_pred_test)

# Print metrics
print("Linear Regression Model")
print("R2 Train:", r2_score_train)
print("R2 Test:", r2_score_test)
print("MAE Train:", mae_train)
print("MAE Test:", mae_test)
print("MSE Train:", mse_train)
print("MSE Test:", mse_test)
print("RMSE Train:", np.sqrt(mse_train))
print("RMSE Test:", np.sqrt(mse_test))

Linear Regression Model
R2 Train: 0.9632873360845152
R2 Test: 0.8200803756773356
MAE Train: 0.20195712251573553
MAE Test: 0.4670472912547392
MSE Train: 0.12821760572939414
MSE Test: 0.5989676062999448
RMSE Train: 0.35807486051019294
RMSE Test: 0.7739299750623081


**Conculsion** : Both the with target transformation or with out target transformation the model is giving similar results. So we can go with the model without target transformation.